# 태그 추출기 — 원하는 태그 · 시간대로 뽑아서 바로 확인

`Log_Extractor.LogExtractor` 를 그대로 불러와서 쓴다(`Custom_Tag_Extractor.py` 와
같은 방식). 아래 순서로 쓰면 된다.

1. **2. 파라미터** 셀에서 `TAGS`(태그 이름 목록) 와 `START_TIME`/`END_TIME`(시간대)을 바꾼다.
2. **3. 추출 실행** 셀을 돌린다 — InfluxDB 에서 뽑아서 `df` 로 받는다.
3. **4. pandas 로 확인** 셀들로 모양/결측치/그래프를 훑어본다.
4. **5. 저장** 셀을 돌리면 `extracted_csv/` 폴더에 CSV 로 떨어진다.

**시간 표기법** (`Log_Extractor._parse_time` 기준):
- 상대시간: `"-1d"`(하루 전), `"-6h"`(6시간 전), `"-30m"`
- 절대시간(KST 로 해석): `"2026-09-10 06:00:00"`
- `"now()"` — 지금

**주의**: `target_tags` 를 비우면 안 된다 — 서버 전체 시리즈(약 1만개)를 피벗하게
되어 현장 DB 에 부하가 크다(`Log_Extractor.get_data` 의 방어 코드 참고).

## 1. 모듈 로드

In [ ]:
import os
import sys
from datetime import datetime

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

# 이 노트북은 parkkt/ 에 있고, Log_Extractor.py 는 한 단계 위(레포 루트)에 있다.
# .env 도 같은 위치(레포 루트)에서 읽는다.
ROOT_DIR = os.path.dirname(os.getcwd())
sys.path.insert(0, ROOT_DIR)

from Log_Extractor import LogExtractor

ENV_PATH = os.path.join(ROOT_DIR, ".env")
SAVE_DIR = os.path.join(os.getcwd(), "extracted_csv")

pd.set_option("display.width", 200)
print(f"ROOT_DIR = {ROOT_DIR}")
print(f"ENV_PATH = {ENV_PATH}  (존재={os.path.exists(ENV_PATH)})")
print(f"SAVE_DIR = {SAVE_DIR}")

## 2. 파라미터 — 여기만 바꿔서 쓰면 된다

`TAGS` 는 InfluxDB 에 올라가는 **태그 이름 그대로** 적는다(공백·한글·특수문자
그대로 가능 — `Log_Extractor` 가 정규식 이스케이프까지 처리한다). 아래는 예시로
P1 탱크의 heater/exchanger/feeding 태그와 wagon 카운터를 넣어 뒀다.

In [ ]:
TAGS = [
    "워킹 Tank 1 BACK Heater",
    "워킹 Tank 1 BACK Cooling SOL",
    "워킹 Tank 1 BACK Exchanger SOL",
    "워킹 Tank 1 BACK Feeding SOL",
    "Now_Actual_Wagon_Num",
]

START_TIME = "-1d"       # 상대시간("-6h", "-30m") 또는 "2026-09-10 06:00:00"(KST)
END_TIME = "now()"
FILE_PREFIX = "Custom"   # 저장 파일명 접두어 (예: P1_Heater_Check)

## 3. 추출 실행

In [ ]:
extractor = LogExtractor(env_path=ENV_PATH)
df = extractor.get_data(start_time=START_TIME, end_time=END_TIME, target_tags=TAGS)
df.shape

## 4. pandas 로 확인

뽑히자마자 훑어봐야 할 세 가지 — 모양/앞부분, 결측치, 시간에 따른 흐름.

In [ ]:
print(f"{df.shape[0]:,}행 × {df.shape[1]}컬럼   기간: {df.index.min()} ~ {df.index.max()}")
df.head(10)

In [ ]:
df.describe().T

In [ ]:
na_count = df.isna().sum()
na_count = na_count[na_count > 0]
if na_count.empty:
    print("결측치 없음")
else:
    print("컬럼별 결측치 개수:")
    print(na_count.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df.plot(ax=ax, linewidth=0.8)
ax.set_title(f"추출 구간 전체 태그 ({df.index.min()} ~ {df.index.max()})")
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()

## 5. 저장 — `extracted_csv/` 폴더에 CSV 로

In [ ]:
if df.empty:
    print("⚠️ 저장할 데이터가 없습니다.")
else:
    os.makedirs(SAVE_DIR, exist_ok=True)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    file_path = os.path.join(SAVE_DIR, f"{FILE_PREFIX}_{timestamp}.csv")
    df.to_csv(file_path, encoding="utf-8-sig")
    print(f"💾 저장 완료: {file_path}")